# Phase 6.5 shard 15 (forest65)

Runs **108 cells** of the frozen Phase 6.5 manifest (`G3-PHASE65-v1`), covering: `causal_drf`, `causal_drf_log`, `causal_drf_retn`, `drf`, `drf_log`.

This shard runs the R forest baselines, including the two adversarial controls (log geometry and bandwidth retune). The setup cell installs R, the pinned `drf` 1.3.1, and the authors' causal-clean package at the frozen commit; fifteen to twenty-five minutes.

Estimated single-threaded compute on the reference machine is about **87 minutes**. Colab cores are slower, so allow two to three times that, plus any install time above. This fits comfortably inside a nine hour session.

**Run every cell in order.** The last cell downloads a `.zip`; collect every shard's zip into `results/phase65/colab_shards/` (logs into `results/manifests/`) and run `python research/run_phase65.py merge`.


In [ ]:
# Thread pinning MUST happen before NumPy or SciPy are imported.
# OpenMP sizes its pool at initialisation, so setting these
# afterwards is silently ineffective.
import os
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS',
           'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS',
           'VECLIB_MAXIMUM_THREADS', 'R_NUM_THREADS'):
    os.environ[_v] = '1'
print('threads pinned to 1')

## 1. Clone the repository at the pinned commit

Remote `https://github.com/hugogobato/wasserstein-causal-forests.git`, commit `5f3b2ad66b0b`. After checkout the notebook asserts the frozen manifest checksum, so a clone of anything but the generating commit fails here rather than mid-run.

In [ ]:
import subprocess, pathlib, os, sys, json, hashlib

REPO = 'https://github.com/hugogobato/wasserstein-causal-forests.git'
COMMIT = '5f3b2ad66b0b05c8f8fed19e0d849d75a206cbe8'
EXPECTED_CHECKSUM = '4e28d308ca99cde4c81379524fc4492a15b38f029b449899b0a307b6c0ace110'

workdir = pathlib.Path('/content/wcf')
if not workdir.exists():
    subprocess.run(['git', 'init', '-q', str(workdir)], check=True)
    subprocess.run(
        ['git', '-C', str(workdir), 'remote', 'add', 'origin', REPO],
        check=True,
    )
# A shallow fetch of the exact commit: nothing else is downloaded.
    subprocess.run(
        ['git', '-C', str(workdir), 'fetch', '-q', '--depth', '1',
         'origin', COMMIT], check=True,
    )
    subprocess.run(
        ['git', '-C', str(workdir), 'checkout', '-q', 'FETCH_HEAD'],
        check=True,
    )
os.chdir(workdir)
sys.path.insert(0, str(workdir / 'src'))
os.environ['WCF_CAUSAL_DRF_R_LIB'] = '/content/wcf/results/Rlib/causal_drf'

manifest = json.load(open(
    'results/manifests/phase65_manifest.json', encoding='utf-8'
))
checksum = hashlib.sha256(
    json.dumps(manifest['cells'], sort_keys=True).encode('utf-8')
).hexdigest()
assert checksum == EXPECTED_CHECKSUM, (
    'the cloned manifest does not match the frozen grid: '
    f'{checksum} != {EXPECTED_CHECKSUM}'
)
print('repo ready at commit ' + COMMIT[:12] + '; '
      + str(manifest['n_cells']) + ' frozen cells verified')

## 2. Dependencies

In [ ]:
# This group runs the R forest baselines, including Causal-DRF
# through the authors' causal-clean package at the frozen commit.
# Expect fifteen to twenty-five minutes for this cell.
%%bash
set -e
apt-get -qq update > /dev/null 2>&1
apt-get -qq install -y r-base r-base-dev libcurl4-openssl-dev libssl-dev libxml2-dev > /dev/null 2>&1
Rscript -e 'options(Ncpus=2); install.packages(c("Rcpp","RcppEigen","jsonlite","remotes","transport"), repos="https://cloud.r-project.org", quiet=TRUE)'
Rscript -e 'options(Ncpus=2); install.packages("https://cran.r-project.org/src/contrib/Archive/drf/drf_1.3.1.tar.gz", repos=NULL, type="source", quiet=TRUE)' || Rscript -e 'options(Ncpus=2); install.packages("drf", repos="https://cloud.r-project.org", quiet=TRUE)'
mkdir -p results/Rlib/causal_drf
Rscript -e 'options(Ncpus=2); .libPaths(c("results/Rlib/causal_drf",.libPaths())); remotes::install_github("herbps10/drf", ref="0a1a508444176b5b1553f13e832be93a374b0af2", lib="results/Rlib/causal_drf", upgrade="never", quiet=TRUE)'
Rscript -e 'cat("drf", as.character(packageVersion("drf")), "ready\n")'
echo 'setup complete'


## 3. This shard's cells

In [ ]:
import json, collections
SHARD_INDEX = 15
CELLS = json.loads('''[{"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "223d80b7aa58d64c", "test_seed": 900003}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "f822d1085207d80c", "test_seed": 900003}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "b0109398e8277d28", "test_seed": 900003}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 4000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "0973a6cdc863c101", "test_seed": 900003}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "14a9f475df9ac733", "test_seed": 900003}, {"grid": "c_scaling", "dgp": "IC0", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "cc6021048c90e95d", "test_seed": 900003}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "0cbd2a9bf45152a7", "test_seed": 900003}, {"grid": "c_scaling", "dgp": "IC1", "n_train": 2000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "bcccedc126492c62", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 3, "cell_key": "69a9745f1e8ea3b5", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 8, "cell_key": "841e8373991b89ec", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 3, "cell_key": "6c07ac7149b97f37", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 8, "cell_key": "7bb2aaf25caf4bbe", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 3, "cell_key": "e545b9c0b9a6a9e3", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 8, "cell_key": "784986381866319e", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 3, "cell_key": "31cc4939313858f5", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 8, "cell_key": "0c69791e412ad868", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 3, "cell_key": "ce431611ef2ad5a4", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 8, "cell_key": "ab4335ee0da71cd4", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 3, "cell_key": "fbb8957a5ac31939", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 8, "cell_key": "4e34552a200a3410", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 3, "cell_key": "e23b112b7002de5e", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 8, "cell_key": "4578d45a4b502cdd", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 3, "cell_key": "bae3b36ba2a3ab54", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_retn", "seed": 8, "cell_key": "63237511b1f1f7e2", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 3, "cell_key": "05b999a21b94a02b", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 8, "cell_key": "a7c91da520872e5c", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 3, "cell_key": "a45dabb630f09923", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 8, "cell_key": "8167d9c3231f6e60", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 3, "cell_key": "7e8c78979332bb8b", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 8, "cell_key": "996b6161b2fa71a0", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 3, "cell_key": "276b23cfd1ed5333", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 8, "cell_key": "796d0014ca8dba85", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 3, "cell_key": "6f76ad3a76c75d5f", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 8, "cell_key": "ac03b80f02d0ca03", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 3, "cell_key": "ff76fa6cccab2735", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 8, "cell_key": "0ed1d34993a1cfb8", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 3, "cell_key": "fc4ddea93a334f50", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 8, "cell_key": "f7fb0835fb8acb47", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 3, "cell_key": "0a0d336c5ad7f0a4", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 8, "cell_key": "18e4f9fbd7242521", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "a41fcdacb1cf9551", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "2cfbb04a062fe89c", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "f35d03cc203218d7", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAskew", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "730a695c2df860d7", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "5707f4cbf6c3a7eb", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "41294c003f01f9f4", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "1cb6e483bd20a588", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DArand", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "6294eb72802688fc", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "7d93dd14b84eee1b", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "e10e1d644e120be6", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "d7451bdf9201ff22", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAunit", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "c900fd874b989fd2", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "082dbbf82b63731e", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "49dcd926965337f7", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "3aa6b8461f74a7d7", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAref", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "5d080851369888a3", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "abfc96c5866f1abc", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "424bf6e28ec99ab6", "test_seed": 900008}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "e36ed106b9b53e4d", "test_seed": 900003}, {"grid": "d_ablation", "dgp": "DAdim", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "7f282c4fdeb0956e", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "f748d4bda45eeb58", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "026a295466e65d1a", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "5b37398fb3581fda", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "fa544e0e32742722", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "97b6c24fc0760add", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "73989f331d758749", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "ec176c6887c18b0b", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "b45fbac4fdff55be", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "96919b2de1de5d5e", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "14b8c7fcc8f352c3", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "5e1492317f96af0d", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "30fb8647daecba44", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "da734a07bcbfba09", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "f3bb3d44ed780d74", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "dfee4578df14246f", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 1000, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "893843a3d58c211c", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 3, "cell_key": "d755497fbe71eb6a", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 8, "cell_key": "9de4d7e25c0f010f", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 3, "cell_key": "20416df92f6b97c9", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 8, "cell_key": "91ab6f888560e418", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 3, "cell_key": "04f9395acce1008d", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 8, "cell_key": "f120be5c527e5fa8", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 3, "cell_key": "9745125783802dcc", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 8, "cell_key": "e99d3e057fffaa83", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 3, "cell_key": "cef90a8760acaec8", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 8, "cell_key": "d7dbd02bfea5ef6a", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 3, "cell_key": "2cc80efa8b53c41f", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 8, "cell_key": "93e693d9343e1aef", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 3, "cell_key": "6c9595b3b301d9d6", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf_log", "seed": 8, "cell_key": "e24a7df46b7a93f1", "test_seed": 900008}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 3, "cell_key": "0f482a75ff03a4de", "test_seed": 900003}, {"grid": "c_controls", "dgp": "IC3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf_log", "seed": 8, "cell_key": "b03fe6adb4be06ef", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "472380113ec410e4", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "62957bd1cb0d98fa", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "03fb57946f7d86df", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI0", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "d1be5a62da94a259", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "ae1270b69cad6e78", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "7bc5624dd437defb", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "a5c7aef10fb9f94b", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI1", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "c76fd65a19a21083", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "b79aa283905cd351", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "cd1700b16d691b3f", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "06171f574762d731", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI2", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "927df2b4e56919b1", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 3, "cell_key": "6bda8007c2ba3c89", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "causal_drf", "seed": 8, "cell_key": "3feb824485171893", "test_seed": 900008}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 3, "cell_key": "ddd2f796a0bf43e6", "test_seed": 900003}, {"grid": "e_zi", "dgp": "ZI3", "n_train": 500, "n_grid": 25, "n_particles": 10, "method": "drf", "seed": 8, "cell_key": "c298825fba72a27c", "test_seed": 900008}]''')
print(f'{len(CELLS)} cells in this shard')
for key, count in sorted(collections.Counter(
        (c['grid'], c['dgp'], c['method'])
        for c in CELLS).items()):
    print(f'  {key[0]:12s} {key[1]:8s} {key[2]:18s} {count}')

## 4. Bandwidth-selection pilot (preregistered)

This shard contains `causal_drf_retn` cells, so it first runs the selection pilot on seeds 100 and 101, outside every decisive range, and freezes the multipliers document. The rule picks the candidate with the best held-out energy score; oracle truth is never read.

In [ ]:
from pathlib import Path
import json, numpy as np
from wasserstein_causal_forests.g3.dgps import build_dgp
from wasserstein_causal_forests.g3.phase65_methods import (
    BANDWIDTH_CANDIDATES, SELECTION_SEEDS, select_bandwidth_multiplier,
)

keys = sorted({(c['dgp'], c['n_train']) for c in CELLS
               if c['method'] == 'causal_drf_retn'})
multipliers = {}
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)
for dgp_name, n_train in keys:
    dgp = build_dgp(dgp_name, 25)
    best, means = select_bandwidth_multiplier(
        dgp, n_train, seeds=SELECTION_SEEDS,
        candidates=BANDWIDTH_CANDIDATES, cache_directory=cache,
    )
    multipliers[f'{dgp_name}|{n_train}'] = best
    scores = {str(k): round(v, 5) for k, v in means.items()}
    print(f'{dgp_name} n={n_train}: multiplier {best}  scores {scores}',
          flush=True)

document = {
    'rule': 'held-out energy score, pilot seeds 100 and 101, '
            'candidates ' + repr(BANDWIDTH_CANDIDATES),
    'multipliers': multipliers,
}
path = Path('/content/wcf/results/manifests/'
            'phase65_bandwidth_selection.json')
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(document, indent=2))
print('froze', path)

## 5. Run

In [ ]:
import time
from pathlib import Path
from wasserstein_causal_forests.g3.manifest import Cell
from wasserstein_causal_forests.g3.runner import run_shard

cells = [Cell(**{k: v for k, v in item.items()
                 if k not in ('cell_key', 'test_seed')})
         for item in CELLS]

out = Path('/content/wcf/results/phase65/colab_shards')
out.mkdir(parents=True, exist_ok=True)
log = Path(f'/content/wcf/results/manifests/phase65_execution_log_{SHARD_INDEX:03d}.jsonl')
log.parent.mkdir(parents=True, exist_ok=True)
cache = Path('/content/wcf/results/rcpp_cache')
cache.mkdir(parents=True, exist_ok=True)

started = time.time()
summary = run_shard(
    cells,
    out / f'shard_{SHARD_INDEX:03d}.parquet',
    cache_directory=cache,
    log_path=log,
    manifest_contract_id='G3-PHASE65-v1',
)
print(json.dumps(summary, indent=2))
print(f'elapsed {(time.time() - started) / 60:.1f} min')

## Check

Every cell must appear exactly once, as a success or as a failure. Failures are kept and reported at merge time; a seed is never silently replaced.

In [ ]:
import collections
records = [json.loads(line) for line in
           open(log, encoding='utf-8') if line.strip()]
status = collections.Counter(r['status'] for r in records)
print('cells logged:', len(records), '| expected:', len(CELLS))
print('status:', dict(status))
assert len(records) == len(CELLS), 'shard did not finish every cell'
for record in records:
    if record['status'] != 'ok':
        print('  FAILED', record['dgp'], record['method'],
              record['seed'])
slowest = sorted(records, key=lambda r: -r['wall_seconds'])[:5]
print('slowest cells:', [(r['method'], round(r['wall_seconds'], 1))
                         for r in slowest])

## Download the results

In [ ]:
import shutil
bundle = '/content/p65_shard_15_forest65'
staging = Path('/content/bundle')
if staging.exists():
    shutil.rmtree(staging)
staging.mkdir(parents=True)
shutil.copy(out / f'shard_{SHARD_INDEX:03d}.parquet', staging)
if log.exists():
    shutil.copy(log, staging)
output_file = shutil.make_archive(bundle, 'zip', staging)
print('bundle:', output_file,
      f'({os.path.getsize(output_file) / 1e6:.2f} MB)')

try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)